# M8_v1c Kaggle Experiment Runner

Notebook nay dong goi workflow `M8_v1c` thanh cac buoc co the chay tuan tu:

1. Kiem tra repo va environment
2. Prepare experiment suite
3. Mine XAI metric policy cho `E3`
4. Export mined policy sang runtime config
5. Preflight `E1-E5`
6. Execute training cho cac thí nghiem duoc chon

Notebook duoc thiet ke theo huong an toan:

- cell execute mac dinh `ENABLE_EXECUTE = False`
- ban can chu dong bat execute truoc khi train that
- cac command deu in ra de de audit

## Cac duong dan chinh

- Repo root: `REPO_ROOT`
- Baseline checkpoint: `experements/yolov8_p2/yolov8_p2_run/weights/best.pt`
- Dataset yaml: `data/YOLO_format/data.yaml`
- Model yaml: `config/yolov8s-p2.yaml`
- Mined runtime config: `configs/method/m8_v1c_runtime_scale_policy_from_metric_candidate.yaml`
- Command pack: `artifacts/m8_v1c_command_pack/m8_v1c_command_pack.md`

In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
from pathlib import Path

try:
    REPO_ROOT = Path('/kaggle/working/XAI-small_object_detection')
    if not REPO_ROOT.exists():
        REPO_ROOT = Path.cwd()
except Exception:
    REPO_ROOT = Path.cwd()

PYTHON_BIN = REPO_ROOT / '.venv' / 'bin' / 'python'
if not PYTHON_BIN.exists():
    PYTHON_BIN = Path('python')

DATA_YAML = REPO_ROOT / 'data' / 'YOLO_format' / 'data.yaml'
MODEL_YAML = REPO_ROOT / 'config' / 'yolov8s-p2.yaml'
BASELINE_CKPT = REPO_ROOT / 'experements' / 'yolov8_p2' / 'yolov8_p2_run' / 'weights' / 'best.pt'
MINE_OUTPUT_DIR = REPO_ROOT / 'artifacts' / 'm8_v1c_xai_metric_policy'
MINED_RUNTIME_CONFIG = REPO_ROOT / 'configs' / 'method' / 'm8_v1c_runtime_scale_policy_from_metric_candidate.yaml'
EXPERIMENT_PROJECT = REPO_ROOT / 'experements' / 'm8_v1c_runtime_train'

DEFAULT_EPOCHS = 100
DEFAULT_IMGSZ = 640
DEFAULT_BATCH = 16
DEFAULT_DEVICE = '0'
DEFAULT_WORKERS = 2
DEFAULT_PATIENCE = 20
DEFAULT_SEED = 42

print('REPO_ROOT =', REPO_ROOT)
print('PYTHON_BIN =', PYTHON_BIN)
print('DATA_YAML exists =', DATA_YAML.exists())
print('MODEL_YAML exists =', MODEL_YAML.exists())
print('BASELINE_CKPT exists =', BASELINE_CKPT.exists())

In [ ]:
os.chdir(REPO_ROOT)

def run_cmd(cmd: str, check: bool = True):
    print('\n$ ' + cmd)
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}')
    return result

def quote(pathlike):
    return shlex.quote(str(pathlike))

def train_cmd(method_config: Path, name: str, mode: str = '--dry-run') -> str:
    return ' '.join([
        quote(PYTHON_BIN),
        'scripts/train_m8_v1c_automated_policy_mining.py',
        '--data', quote(DATA_YAML),
        '--model', quote(MODEL_YAML),
        '--method-config', quote(method_config),
        '--epochs', str(DEFAULT_EPOCHS),
        '--imgsz', str(DEFAULT_IMGSZ),
        '--batch', str(DEFAULT_BATCH),
        '--device', DEFAULT_DEVICE,
        '--workers', str(DEFAULT_WORKERS),
        '--patience', str(DEFAULT_PATIENCE),
        '--project', quote(EXPERIMENT_PROJECT),
        '--name', name,
        '--seed', str(DEFAULT_SEED),
        mode,
    ])

def load_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

## 0. Sanity check repo

In [ ]:
run_cmd('pwd')
run_cmd('ls -lah')
run_cmd('find configs/method -maxdepth 1 -type f | sort')

## 1. Prepare experiment suite

In [ ]:
prepare_suite_cmd = f"{quote(PYTHON_BIN)} scripts/prepare_m8_v1c_experiment_suite.py"
run_cmd(prepare_suite_cmd)
suite_manifest = REPO_ROOT / 'artifacts' / 'm8_v1c_experiment_suite' / 'm8_v1c_experiment_suite_manifest.json'
print(load_json(suite_manifest))

## 2. Mine policy cho E3

Dat `RUN_MINE_POLICY = True` neu ban muon chay lai mining. Neu da co artifact policy candidate roi, co the bo qua.

In [ ]:
RUN_MINE_POLICY = False

mine_cmd = ' '.join([
    quote(PYTHON_BIN),
    'scripts/analyze_m8_v1c_xai_metric_policy.py',
    '--model', quote(BASELINE_CKPT),
    '--model-config', quote(MODEL_YAML),
    '--data', quote(DATA_YAML),
    '--split', 'val',
    '--imgsz', str(DEFAULT_IMGSZ),
    '--max-images', '64',
    '--device', 'cpu',
    '--output-dir', quote(MINE_OUTPUT_DIR),
])

if RUN_MINE_POLICY:
    run_cmd(mine_cmd)
else:
    print('Skipping mine step. Set RUN_MINE_POLICY = True to enable.')

## 3. Export mined policy sang runtime config

In [ ]:
policy_candidate = MINE_OUTPUT_DIR / 'm8_v1c_metric_policy_candidate.yaml'
export_cmd = ' '.join([
    quote(PYTHON_BIN),
    'scripts/export_m8_v1c_metric_policy_to_runtime_config.py',
    '--policy-candidate', quote(policy_candidate),
    '--output', quote(MINED_RUNTIME_CONFIG),
])
run_cmd(export_cmd)

## 4. Prepare command pack

In [ ]:
command_pack_cmd = f"{quote(PYTHON_BIN)} scripts/prepare_m8_v1c_command_pack.py"
run_cmd(command_pack_cmd)
command_pack_summary = REPO_ROOT / 'artifacts' / 'm8_v1c_command_pack' / 'm8_v1c_command_pack_summary.json'
print(load_json(command_pack_summary))

## 5. Preflight cho E1-E5

In [ ]:
EXPERIMENTS = {
    'E1': REPO_ROOT / 'configs' / 'method' / 'm8_v1c_runtime_scale_policy_identity.yaml',
    'E2': REPO_ROOT / 'configs' / 'method' / 'm8_v1c_runtime_scale_policy_manual_reference.yaml',
    'E3': REPO_ROOT / 'configs' / 'method' / 'm8_v1c_runtime_scale_policy_from_metric_candidate.yaml',
    'E4': REPO_ROOT / 'configs' / 'method' / 'm8_v1c_runtime_scale_policy_random_seed42.yaml',
    'E5': REPO_ROOT / 'configs' / 'method' / 'm8_v1c_runtime_scale_policy_inverted.yaml',
}

for exp_id, config_path in EXPERIMENTS.items():
    run_cmd(train_cmd(config_path, name=f'{exp_id.lower()}_preflight', mode='--dry-run'))

## 6. Execute training

Mac dinh notebook khong train that. Ban phai dat `ENABLE_EXECUTE = True` va chon `SELECTED_EXPERIMENTS`.

Thu tu uu tien khuyen nghi:

1. `E1`
2. `E3`
3. `E2`
4. `E4`
5. `E5`

In [ ]:
ENABLE_EXECUTE = False
SELECTED_EXPERIMENTS = ['E1', 'E3']

if ENABLE_EXECUTE:
    for exp_id in SELECTED_EXPERIMENTS:
        config_path = EXPERIMENTS[exp_id]
        run_cmd(train_cmd(config_path, name=f'{exp_id.lower()}_execute', mode='--execute'))
else:
    print('Execute is disabled. Set ENABLE_EXECUTE = True to train.')
    for exp_id in SELECTED_EXPERIMENTS:
        print(train_cmd(EXPERIMENTS[exp_id], name=f'{exp_id.lower()}_execute', mode='--execute'))

## 7. Ghi nho sau moi run

Sau moi execute, ban nen luu lai:

- ten thí nghiem (`E1-E5`)
- runtime config da dung
- seed
- `results.csv`
- checkpoint tot nhat
- nhan xet ngan: tot hon baseline / ngang baseline / kem hon baseline